# 1. What you'll learn

By the end of this notebook, you will be able to:

- choose metrics that reflect the cost of false positives and false negatives;
- explain why accuracy can be nearly useless on an imbalanced problem;
- read a reliability diagram and separate ranking from calibrated probability.

**The one question this notebook answers:** When only a few cases are
positive, what does “good” model performance actually mean?

# 2. Setup

We seed data generation, splitting, and every estimator that uses randomness.
Think of the seed as a lab notebook entry: it does not make one experiment
universally true, but it lets you reproduce this exact experiment before
trying other seeds as a robustness check. Versions are printed because metric
defaults and estimator behavior can change across releases.

In [ ]:
from pathlib import Path
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.datasets import make_classification
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, average_precision_score,
    brier_score_loss, f1_score, precision_recall_curve, precision_score,
    recall_score, roc_auc_score, roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 0
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

We generate 4,000 screening cases with eight measured features and a 6%
positive rate. The target `needs_review` is 1 for the rare cases we hope to
find. All columns are numeric synthetic measurements; six contain signal,
two are redundant or noisy. Because we know how the data was generated, we
can distinguish a metric lesson from a data-quality mystery.

If only 6 of 100 cases are positive, saying “negative” 100 times is 94%
accurate. That arithmetic is the trap this notebook will make visible.

In [ ]:
X, y = make_classification(
    n_samples=4_000,
    n_features=8,
    n_informative=6,
    n_redundant=1,
    weights=[0.94, 0.06],
    class_sep=1.15,
    flip_y=0.01,
    random_state=SEED,
)
feature_names = [f"measurement_{i}" for i in range(X.shape[1])]
frame = pd.DataFrame(X, columns=feature_names)
frame["needs_review"] = y

print("Shape:", frame.shape)
print("Class counts:\n", frame["needs_review"].value_counts().sort_index())
print("Positive rate:", round(frame["needs_review"].mean(), 4))
print("\nFirst three rows:\n", frame.head(3).round(3))

# 4. Explore

The first plot shows the base rate. Base rate is not background trivia: it
sets the precision of random alerts and the accuracy of the majority guess.
A tiny positive bar tells you to distrust accuracy before any model exists.

In [ ]:
counts = frame["needs_review"].value_counts().sort_index()
ax = counts.plot(kind="bar", color=["#8aa6a3", "#d38b45"], figsize=(6, 3.5))
ax.set(title="A rare positive class", xlabel="Needs review (0 = no, 1 = yes)", ylabel="Cases")
ax.bar_label(ax.containers[0])
plt.tight_layout()
plt.show()

**Takeaway:** A classifier can exceed 93% accuracy by predicting no positive
cases. Accuracy alone cannot tell useful detection from total inaction.

Now compare one measurement by class. The positive distribution shifts, but
overlap remains. Overlap creates unavoidable false positives and false
negatives, so the decision threshold must encode a trade-off.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
sns.kdeplot(data=frame, x="measurement_0", hue="needs_review", fill=True, common_norm=False, alpha=0.35, ax=ax)
ax.set_title("Signal exists, but the classes overlap")
plt.tight_layout()
plt.show()

**Takeaway:** No clean boundary exists on one feature. Changing a threshold
will exchange some missed positives for some unnecessary reviews.

A two-feature view makes the same point geometrically. We sample rows only
for readability; the sample is selected with the fixed seed, so the picture
remains reproducible.

In [ ]:
sample = frame.sample(700, random_state=SEED)
fig, ax = plt.subplots(figsize=(6.5, 4.5))
sns.scatterplot(data=sample, x="measurement_0", y="measurement_1", hue="needs_review", alpha=0.65, ax=ax)
ax.set_title("Rare positives occupy overlapping regions")
plt.tight_layout()
plt.show()

**Takeaway:** A model will produce a graded score, not discover a perfectly
separable island. Metrics translate that score into different views of value.

# 5. Prepare

We reserve 20% for final testing, then split the remaining 80% into 60%
training and 20% validation. Stratification preserves the rare positive rate
in each part. Without it, a small validation set might accidentally contain
too few positives for stable recall or PR AUC.

Scaling stays inside each pipeline, so its means and standard deviations are
learned from training folds only. Fitting a scaler before the split would let
held-out feature distributions influence training.

In [ ]:
development, test = train_test_split(
    frame, test_size=0.20, stratify=frame["needs_review"], random_state=SEED
)
train, validation = train_test_split(
    development, test_size=0.25, stratify=development["needs_review"], random_state=SEED
)

X_train, y_train = train[feature_names], train["needs_review"]
X_validation, y_validation = validation[feature_names], validation["needs_review"]
X_test, y_test = test[feature_names], test["needs_review"]

for name, target in [("train", y_train), ("validation", y_validation), ("test", y_test)]:
    print(f"{name:10s} n={len(target):4d}, positives={target.sum():3d}, rate={target.mean():.3f}")

# 6. Train

We fit four reference points. The dummy model predicts the prior. Plain
logistic regression usually preserves probability meaning reasonably well.
Class weighting tells the model that rare positives matter more, often
improving recall while distorting raw probabilities. Sigmoid calibration
then learns a mapping from weighted scores back to observed frequencies,
using cross-validation inside the training set.

In [ ]:
def logistic_pipeline(class_weight=None):
    return Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(
            C=1.0, class_weight=class_weight, max_iter=1_000, random_state=SEED
        )),
    ])

dummy = DummyClassifier(strategy="prior", random_state=SEED)
plain = logistic_pipeline()
weighted = logistic_pipeline(class_weight="balanced")
calibrated = CalibratedClassifierCV(
    estimator=logistic_pipeline(class_weight="balanced"), method="sigmoid", cv=3
)

for model in (dummy, plain, weighted, calibrated):
    model.fit(X_train, y_train)

print("Training rows:", X_train.shape)
print("Fitted: prior dummy, plain logistic, weighted logistic, calibrated weighted logistic")

# 7. Evaluate

Each metric asks a different question. Precision is “of the alerts, how many
were real?” Recall is “of the real positives, how many did we find?” ROC AUC
measures ranking across thresholds. PR AUC emphasizes the rare positive
class. Brier loss is the mean squared probability error, so lower is better.
We include accuracy to expose, rather than hide, its behavior.

In [ ]:
models = {"dummy": dummy, "plain": plain, "weighted": weighted, "calibrated": calibrated}

def metric_row(model, X_part, y_part, threshold=0.5):
    probability = model.predict_proba(X_part)[:, 1]
    prediction = (probability >= threshold).astype(int)
    return {
        "accuracy": accuracy_score(y_part, prediction),
        "precision": precision_score(y_part, prediction, zero_division=0),
        "recall": recall_score(y_part, prediction, zero_division=0),
        "roc_auc": roc_auc_score(y_part, probability),
        "pr_auc": average_precision_score(y_part, probability),
        "brier_loss": brier_score_loss(y_part, probability),
    }

test_metrics = pd.DataFrame({
    name: metric_row(model, X_test, y_test) for name, model in models.items()
}).T
print(test_metrics.round(3))

A confusion matrix turns the chosen 0.5 threshold into case counts. Reading
the bottom-left cell tells you how many true positives were missed; the
top-right tells you how many extra reviews were created. Counts are often
easier to connect to staffing or safety cost than a percentage.

In [ ]:
calibrated_probability = calibrated.predict_proba(X_test)[:, 1]
calibrated_prediction = (calibrated_probability >= 0.5).astype(int)
ConfusionMatrixDisplay.from_predictions(y_test, calibrated_prediction, cmap="Blues", colorbar=False)
plt.title("Calibrated model at threshold 0.50")
plt.tight_layout()
plt.show()

The combined figure separates three properties. ROC asks about ranking over
all classes. Precision–recall focuses on useful positive retrieval. The
reliability diagram groups predictions into bins: if cases predicted near
0.30 are positive about 30% of the time, that point lies near the diagonal.

In [ ]:
probability_by_model = {
    name: model.predict_proba(X_test)[:, 1]
    for name, model in {"plain": plain, "weighted": weighted, "calibrated": calibrated}.items()
}

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
colors = {"plain": "#176b66", "weighted": "#d38b45", "calibrated": "#76558f"}
for name, probability in probability_by_model.items():
    fpr, tpr, _ = roc_curve(y_test, probability)
    precision, recall, _ = precision_recall_curve(y_test, probability)
    observed, predicted = calibration_curve(y_test, probability, n_bins=8, strategy="quantile")
    axes[0].plot(fpr, tpr, label=name, color=colors[name])
    axes[1].plot(recall, precision, label=name, color=colors[name])
    axes[2].plot(predicted, observed, marker="o", label=name, color=colors[name])

axes[0].plot([0, 1], [0, 1], "--", color="#8a9290")
axes[0].set(title="ROC: ranking", xlabel="False-positive rate", ylabel="True-positive rate")
axes[1].axhline(y_test.mean(), ls="--", color="#8a9290", label="prevalence")
axes[1].set(title="Precision–recall", xlabel="Recall", ylabel="Precision")
axes[2].plot([0, 1], [0, 1], "--", color="#8a9290", label="perfect calibration")
axes[2].set(title="Reliability", xlabel="Mean predicted probability", ylabel="Observed frequency")
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(Path.cwd() / "key_figure.png", dpi=160, bbox_inches="tight")
plt.show()

**Takeaway:** Class weighting can preserve ranking while pushing probabilities
too high. Calibration repairs probability meaning; it does not magically
create new ranking information.

# 8. Break it

First we deliberately choose by accuracy alone. The dummy model's score looks
excellent because negatives dominate, yet its recall is zero. Then we show a
legitimate use of validation data: choose the threshold that maximizes F1 on
validation, freeze it, and report the result on test. Choosing the threshold
on test would leak the answer to a model-selection question.

In [ ]:
dummy_row = test_metrics.loc["dummy"]
print(f"Accuracy-only choice: dummy accuracy={dummy_row['accuracy']:.3f}, recall={dummy_row['recall']:.3f}")

validation_probability = calibrated.predict_proba(X_validation)[:, 1]
thresholds = np.linspace(0.05, 0.95, 181)
validation_f1 = [f1_score(y_validation, validation_probability >= t) for t in thresholds]
chosen_threshold = float(thresholds[int(np.argmax(validation_f1))])

default_result = metric_row(calibrated, X_test, y_test, threshold=0.50)
tuned_result = metric_row(calibrated, X_test, y_test, threshold=chosen_threshold)
print(f"Validation-chosen threshold: {chosen_threshold:.2f}")
print(f"Test at 0.50 -> precision={default_result['precision']:.3f}, recall={default_result['recall']:.3f}")
print(f"Test at {chosen_threshold:.2f} -> precision={tuned_result['precision']:.3f}, recall={tuned_result['recall']:.3f}")

**Use this when…** positives are uncommon, decision costs differ, or a score
will be interpreted as a probability. Name the decision first, then select
metrics and thresholds that match it.

**Don't use this when…** you have not defined what a positive means, labels
are delayed or biased, or one aggregate metric would hide performance for
important groups. Fix the measurement problem before polishing the score.

## Try this

1. Change the positive weight from 0.06 to 0.01. Dummy accuracy should rise
   while PR AUC becomes much harder to improve.
2. Replace `method="sigmoid"` with `method="isotonic"`. With limited positive
   data it may become jagged or overfit; compare Brier loss on test.
3. Choose the validation threshold for recall of at least 0.90 instead of
   maximum F1. Expect more true positives and more false alarms.